# Dock a molecule with dockstring and look at it in 3D

This notebook takes a SMILES string, shows it in 3D, loads one of dockstring's built-in DUD-E receptors, docks the molecule with AutoDock Vina (through dockstring) and shows the docked pose in the receptor.

1. Install dockstring, RDKit and py3Dmol
2. Enter a SMILES and view the molecule in 3D
3. Choose a receptor (DRD2, DRD3, ESR1, ESR2, ADRB1 or ADRB2) and view it as a ribbon
4. Dock, report the top score, and view the best pose (sticks) in the receptor (ribbon)

Run the cells from the top. Edit the SMILES in step 2 and the receptor in step 3 to try other molecules and targets.

## 1. Install

This installs dockstring (which bundles AutoDock Vina and the DUD-E receptors), RDKit, py3Dmol and Open Babel. It only needs to run once per session; on Colab it takes about a minute.

In [ ]:
%pip install -q dockstring rdkit py3Dmol openbabel-wheel

Import the libraries and check the setup: RDKit handles the molecule, py3Dmol draws the 3D views, and dockstring does the docking. dockstring needs Open Babel (the `obabel` program, installed with `openbabel-wheel`) to prepare the molecule for docking, so the cell makes sure the copy installed next to this Python is the one used (a system-wide copy can be older or broken), checks it is available, and prints how many built-in receptors dockstring has.

In [ ]:
import os
import shutil
import sys

import py3Dmol
from rdkit import Chem
from dockstring import list_all_target_names, load_target
from dockstring import utils as dsu   # dockstring's own ligand-preparation functions

os.environ["PATH"] = os.path.dirname(sys.executable) + os.pathsep + os.environ["PATH"]   # use the obabel that pip installed next to this Python, not a system copy
assert shutil.which("obabel"), "Open Babel (obabel) was not found on the PATH; dockstring needs it"
print("dockstring has", len(list_all_target_names()), "built-in targets")

## 2. A molecule in 3D

Type a SMILES between the quotes (the default is paracetamol), then run the cell. The molecule is prepared exactly as dockstring prepares it before docking: it is sanitised, protonated at pH 7.4 with Open Babel, given hydrogens and a 3D structure by RDKit (ETKDG embedding), and energy-minimised with MMFF94 (UFF as a fallback). So the structure you see here, with the charge state it has at pH 7.4, is the molecule that will be docked.

In [ ]:
SMILES = "CC(=O)Nc1ccc(O)cc1"  #@param {type:"string"}

# the same preparation dockstring does before it docks (see Target.dock)
mol = dsu.sanitize_mol(dsu.smiles_to_mol(dsu.canonicalize_smiles(SMILES)))
dsu.check_mol(mol)
protonated = dsu.protonate_mol(mol, pH=7.4)                      # Open Babel, pH 7.4
mol3d = dsu.refine_mol_with_ff(dsu.embed_mol(protonated, seed=974528263))   # add Hs, 3D embedding, MMFF94 (UFF as a fallback)
dsu.assign_stereochemistry(mol3d)
print("SMILES as docked (protonated at pH 7.4):", Chem.MolToSmiles(protonated))

view = py3Dmol.view(width=500, height=400)
view.addModel(Chem.MolToMolBlock(mol3d), "mol")
view.setStyle({"stick": {}})
view.zoomTo()
view.show()

## 3. Choose a receptor

dockstring ships prepared DUD-E receptors. Pick one by setting `TARGET` (in Colab, use the drop-down), then run the cell. The receptor file is dockstring's PDBQT, which py3Dmol reads directly. It is drawn as a ribbon coloured from the N- to the C-terminus.

In [ ]:
TARGET = "DRD2"  #@param ["DRD2", "DRD3", "ESR1", "ESR2", "ADRB1", "ADRB2"]

target = load_target(TARGET)
receptor_text = open(target.pdbqt_path).read()   # py3Dmol reads PDBQT directly
n_res = len({l[21:26] for l in receptor_text.splitlines() if l.startswith("ATOM") and l[12:16].strip() == "CA"})
print(f"{TARGET}: {n_res} residues from {target.pdbqt_path}")

view = py3Dmol.view(width=700, height=500)
view.addModel(receptor_text, "pdbqt")
view.setStyle({"cartoon": {"color": "spectrum"}})
view.zoomTo()
view.show()

## 4. Dock

dockstring protonates the molecule at pH 7.4, embeds it, and docks it with AutoDock Vina into the receptor's prepared search box. This takes about a minute. The top score is the best pose's Vina affinity in kcal/mol; more negative is a stronger predicted binder. All poses are saved to an SDF file.

In [ ]:
score, aux = target.dock(SMILES)
assert score is not None, "docking failed for this molecule"

ligand = aux["ligand"]
affinities = aux["affinities"]
print(f"Top docking score against {TARGET}: {score:.1f} kcal/mol")
print("All poses (kcal/mol):", ", ".join(f"{a:.1f}" for a in affinities))

dock_sdf = f"docked_{TARGET}.sdf"
writer = Chem.SDWriter(dock_sdf)
for conf, aff in zip(ligand.GetConformers(), affinities):
    ligand.SetProp("vina_affinity_kcal_mol", f"{aff:.2f}")
    writer.write(ligand, confId=conf.GetId())
writer.close()
print("Poses written to", dock_sdf)

The best pose (the first in the SDF) as sticks, inside the receptor as a ribbon. Drag to rotate, scroll to zoom.

In [ ]:
best_pose = Chem.MolToMolBlock(next(iter(Chem.SDMolSupplier(dock_sdf))))

view = py3Dmol.view(width=800, height=600)
view.addModel(receptor_text, "pdbqt")
view.setStyle({"model": 0}, {"cartoon": {"color": "spectrum", "opacity": 0.7}})
view.addModel(best_pose, "sdf")
view.setStyle({"model": 1}, {"stick": {"radius": 0.25, "colorscheme": "yellowCarbon"}})
view.zoomTo({"model": 1})
view.zoom(0.6)
view.show()